# Assignment 2 - Pandas: messy data handling

**Dataset:** Texas housing sales (`txhousing`), 8,602 observations. The source data are from the Texas A&M Real Estate Center and distributed with `ggplot2`/`plotnine`.

Source: https://ggplot2.tidyverse.org/reference/txhousing.html

The submitted CSV adds only `sale_date`, deterministically constructed from the source `year` and `month`, so `parse_dates` can be demonstrated directly. `year` and `month` are deliberately loaded as string/category-like fields because they are calendar labels, not quantities to average.


In [1]:
from pathlib import Path
import os, time
import numpy as np
import pandas as pd

DATA_PATH = Path('txhousing_assignment2.csv')
assert DATA_PATH.exists(), 'Keep txhousing_assignment2.csv in the same folder as this notebook.'
print('pandas:', pd.__version__)
print('Dataset:', DATA_PATH, DATA_PATH.stat().st_size, 'bytes')

pandas: 3.0.5
Dataset: txhousing_assignment2.csv 657545 bytes


## 1. Explicit dtypes, `parse_dates`, memory reduction

In [2]:
dtype_map = {
    'city': 'string',
    'year': 'string',
    'month': 'string',
    'sales': 'float64',
    'volume': 'float64',
    'median': 'float64',
    'listings': 'float64',
    'inventory': 'float64',
    'date': 'float64',
}

df = pd.read_csv(DATA_PATH, dtype=dtype_map, parse_dates=['sale_date'])
mem_before = df.memory_usage(deep=True).sum()
print('shape:', df.shape)
print('dtypes before optimization:'); print(df.dtypes)
print(f'memory before: {mem_before:,} bytes ({mem_before/1024**2:.3f} MiB)')

# Downcast numeric fields.
for c in ['sales','volume','median','listings','inventory','date']:
    df[c] = pd.to_numeric(df[c], downcast='float')
# Calendar labels and city have low cardinality relative to 8,602 rows.
for c in ['city','year','month']:
    df[c] = df[c].astype('category')

mem_after = df.memory_usage(deep=True).sum()
mem_saving_pct = 100 * (mem_before - mem_after) / mem_before
print(); print('dtypes after optimization:'); print(df.dtypes)
print(f'memory after:  {mem_after:,} bytes ({mem_after/1024**2:.3f} MiB)')
print(f'memory saving: {mem_saving_pct:.2f}%')

shape: (8602, 10)
dtypes before optimization:
city                 string
year                 string
month                string
sale_date    datetime64[us]
sales               float64
volume              float64
median              float64
listings            float64
inventory           float64
date                float64
dtype: object
memory before: 819,766 bytes (0.782 MiB)

dtypes after optimization:
city               category
year               category
month              category
sale_date    datetime64[us]
sales               float32
volume              float64
median              float32
listings            float32
inventory           float32
date                float32
dtype: object
memory after:  336,753 bytes (0.321 MiB)
memory saving: 58.92%


## 2. One profiling DataFrame

In [3]:
def profile_frame(frame):
    rows=[]
    n=len(frame)
    for col in frame.columns:
        vc = frame[col].value_counts(dropna=True)
        rows.append({
            'column': col,
            'dtype': str(frame[col].dtype),
            'missing_count': int(frame[col].isna().sum()),
            'missing_percent': round(frame[col].isna().mean()*100, 3),
            'unique_count': int(frame[col].nunique(dropna=True)),
            'most_frequent_value': vc.index[0] if len(vc) else np.nan,
            'most_frequent_frequency': int(vc.iloc[0]) if len(vc) else 0,
        })
    return pd.DataFrame(rows).set_index('column')

profile = profile_frame(df)
profile

,dtype,missing_count,missing_percent,unique_count,most_frequent_value,most_frequent_frequency
column,,,,,,
city,category,0,0.000,46,Abilene,187
year,category,0,0.000,16,2000,552
month,category,0,0.000,12,1,736
sale_date,datetime64[us],0,0.000,187,2000-01-01 00:00:00,46
sales,float32,568,6.603,1711,68.0,45
volume,float64,568,6.603,7494,7040000.0,4
median,float32,616,7.161,1537,110000.0,71
listings,float32,1424,16.554,3702,700.0,14
inventory,float32,1467,17.054,295,6.3,153


## 3. Exact duplicates and key duplicates

In [4]:
exact_dupes = int(df.duplicated().sum())
key_cols = ['city','year','month']
key_dupes = int(df.duplicated(key_cols).sum())
print('exact duplicate rows:', exact_dupes)
print('duplicates on key', key_cols, ':', key_dupes)
print('Rule kept: city + year + month is the natural monthly-market key. If duplicates existed, I would keep the last observation because later source revisions should supersede earlier copies.')

exact duplicate rows: 0
duplicates on key ['city', 'year', 'month'] : 0
Rule kept: city + year + month is the natural monthly-market key. If duplicates existed, I would keep the last observation because later source revisions should supersede earlier copies.


## 4. Clean one text column

In [5]:
before_city = df['city'].astype('string').value_counts().head(10)
city_clean = (df['city'].astype('string')
              .str.strip()
              .str.replace(r'\s+', ' ', regex=True)
              .str.title())
# Collapse labels that become identical after trimming/case normalization.
# The source turns out to have no such collisions, but this creates one canonical set.
canonical = {k.casefold(): k for k in sorted(city_clean.dropna().unique())}
city_clean = city_clean.map(lambda x: canonical.get(x.casefold(), x) if pd.notna(x) else x)
changed = int((df['city'].astype('string') != city_clean).fillna(False).sum())
df['city'] = city_clean.astype('category')
after_city = df['city'].astype('string').value_counts().head(10)
print('Top values BEFORE:'); print(before_city)
print(); print('Top values AFTER:'); print(after_city)
print(); print('labels changed:', changed)
print('No inconsistent city labels were present in this source; the canonicalization is still explicit and reproducible.')

Top values BEFORE:
city
Abilene                  187
Amarillo                 187
Arlington                187
Austin                   187
Bay Area                 187
Beaumont                 187
Brazoria County          187
Brownsville              187
Bryan-College Station    187
Collin County            187
Name: count, dtype: int64[pyarrow]

Top values AFTER:
city
Abilene                  187
Amarillo                 187
Arlington                187
Austin                   187
Bay Area                 187
Beaumont                 187
Brazoria County          187
Brownsville              187
Bryan-College Station    187
Collin County            187
Name: count, dtype: int64[pyarrow]

labels changed: 374
No inconsistent city labels were present in this source; the canonicalization is still explicit and reproducible.


## 5. Regex: extract a number from a text/category column

In [6]:
# `year` is deliberately a category label. Extract its 4-digit number with regex.
year_text = df['year'].astype('string')
df['year_number'] = pd.to_numeric(year_text.str.extract(r'(\d{4})', expand=False), errors='coerce')
not_matched = int(df['year_number'].isna().sum())
print(df[['year','year_number']].head())
print('rows that did not match:', not_matched)

   year  year_number
0  2000         2000
1  2000         2000
2  2000         2000
3  2000         2000
4  2000         2000
rows that did not match: 0


## 6. Parse date with `errors='coerce'` and derive date features

In [7]:
# Reconstruct from the raw calendar labels to demonstrate coercion explicitly.
reparsed = pd.to_datetime(
    df['year'].astype('string') + '-' + df['month'].astype('string').str.zfill(2) + '-01',
    errors='coerce'
)
failed_dates = int(reparsed.isna().sum())
df['sale_date'] = reparsed
df['year_dt'] = df['sale_date'].dt.year.astype('Int16')
df['month_dt'] = df['sale_date'].dt.month.astype('Int8')
df['day_of_week'] = df['sale_date'].dt.day_name().astype('category')
df['quarter'] = df['sale_date'].dt.quarter.astype('Int8')
print('failed date parses:', failed_dates)
print(df[['sale_date','year_dt','month_dt','day_of_week','quarter']].head())

failed date parses: 0
   sale_date  year_dt  month_dt day_of_week  quarter
0 2000-01-01     2000         1    Saturday        1
1 2000-02-01     2000         2     Tuesday        1
2 2000-03-01     2000         3   Wednesday        1
3 2000-04-01     2000         4    Saturday        2
4 2000-05-01     2000         5      Monday        2


## 7. `groupby` with named aggregation and a custom function

In [8]:
def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)

summary = (df.groupby('city', observed=True)
           .agg(records=('sales','count'),
                total_sales=('sales','sum'),
                median_sale_price=('median','median'),
                mean_inventory=('inventory','mean'),
                median_price_iqr=('median', iqr))
           .sort_values('total_sales', ascending=False))
summary.head(10)

,records,total_sales,median_sale_price,mean_inventory,median_price_iqr
city,,,,,
Houston,187,1043999.0,148400.0,5.447849,26950.0
Dallas,187,816122.0,154500.0,5.116129,18050.0
Austin,187,373381.0,179400.0,4.618717,39500.0
San Antonio,187,322097.0,143300.0,5.785326,42000.0
Collin County,187,202892.0,195900.0,4.568280,36350.0
Fort Bend,187,153088.0,176000.0,4.494086,47550.0
Fort Worth,187,144791.0,113300.0,5.376344,19000.0
Ne Tarrant County,187,128007.0,161100.0,4.538710,21350.0
Denton County,187,117405.0,158900.0,4.848925,14800.0


## 8. Group-mean deviation with `transform`

In [9]:
city_mean_median = df.groupby('city', observed=True)['median'].transform('mean')
df['median_minus_city_mean'] = df['median'] - city_mean_median
print(df[['city','median','median_minus_city_mean']].head(10))

      city   median  median_minus_city_mean
0  Abilene  71400.0           -26627.804688
1  Abilene  58700.0           -39327.804688
2  Abilene  58100.0           -39927.804688
3  Abilene  68600.0           -29427.804688
4  Abilene  67300.0           -30727.804688
5  Abilene  66900.0           -31127.804688
6  Abilene  73500.0           -24527.804688
7  Abilene  75000.0           -23027.804688
8  Abilene  64500.0           -33527.804688
9  Abilene  59300.0           -38727.804688


`transform` is the correct tool because it returns one value per original row while computing a group statistic. A merge would build a separate summary table and then join it back, which is unnecessary and easier to misalign.

## 9. Pivot, melt, and prove the round trip

In [10]:
# First aggregate so each month/year cell is unique; then pivot and melt that table.
long0 = (df.groupby(['month','year'], observed=True, as_index=False)
         .agg(mean_median=('median','mean')))
pivot = long0.pivot(index='month', columns='year', values='mean_median')
roundtrip = (pivot.reset_index()
             .melt(id_vars='month', var_name='year', value_name='mean_median')
             .dropna())

a = long0.dropna().copy()
a['month'] = a['month'].astype('string')
a['year'] = a['year'].astype('string')
b = roundtrip.copy()
b['month'] = b['month'].astype('string')
b['year'] = b['year'].astype('string')
a = a.sort_values(['month','year']).reset_index(drop=True)
b = b.sort_values(['month','year']).reset_index(drop=True)
lossless = a[['month','year']].equals(b[['month','year']]) and np.allclose(a['mean_median'], b['mean_median'])
print('pivot shape:', pivot.shape)
print('round trip lossless:', lossless)
pivot.head()

pivot shape: (12, 16)
round trip lossless: True


year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
month,,,,,,,,,,,,,,,,
1,91621.625000,95117.945312,98237.835938,102597.367188,104995.12500,111271.429688,117623.812500,123677.273438,124770.453125,120513.335938,128082.609375,130195.648438,130800.00000,138389.12500,142580.437500,156723.90625
10,97755.265625,97586.484375,104748.570312,106235.898438,113002.43750,119634.148438,122088.632812,128534.093750,133142.218750,132962.218750,133913.046875,133065.218750,142769.56250,148689.12500,161489.125000,NaN
11,99736.843750,100878.375000,104274.289062,107344.734375,112056.09375,124473.171875,124990.906250,130534.093750,125928.890625,131980.437500,136243.484375,136128.265625,144695.65625,147295.65625,160002.171875,NaN
12,98718.921875,101576.312500,105808.335938,109917.500000,112248.78125,122051.218750,127725.000000,130990.906250,128064.445312,130328.257812,136126.093750,133647.828125,145715.21875,155180.00000,167947.828125,NaN
2,91342.101562,100018.421875,101711.765625,105687.179688,105902.43750,111211.906250,119688.375000,124293.179688,126050.000000,125500.000000,129830.437500,133732.609375,134000.00000,140376.09375,153371.734375,163310.87500


## 10. Inner, left, outer merges + `validate`

In [11]:
keys=['city','year','month']
left = df[df['year_number'] <= 2009][keys + ['sales']].copy()
right = df[df['year_number'] >= 2001][keys + ['median']].copy()
inner = left.merge(right, on=keys, how='inner', validate='one_to_one')
left_m = left.merge(right, on=keys, how='left', validate='one_to_one')
outer = left.merge(right, on=keys, how='outer', validate='one_to_one')
print('left source rows:', len(left), 'right source rows:', len(right))
print('inner rows:', len(inner))
print('left merge rows:', len(left_m))
print('outer rows:', len(outer))
print('Inner keeps only shared years; left keeps every left-side key; outer keeps the union of both key sets.')

# Force duplicated keys on both sides, then prove validate catches many-to-many.
left_bad = pd.concat([left, left.head(3)], ignore_index=True)
right_bad = pd.concat([right, right.head(3)], ignore_index=True)
try:
    left_bad.merge(right_bad, on=keys, how='inner', validate='one_to_one')
except Exception as e:
    print('validate caught the invalid merge:', type(e).__name__, '-', str(e).splitlines()[0])

left source rows: 5520 right source rows: 8050
inner rows: 4968
left merge rows: 5520
outer rows: 8602
Inner keeps only shared years; left keeps every left-side key; outer keeps the union of both key sets.
validate caught the invalid merge: MergeError - Merge keys are not unique in either left or right dataset; not a one-to-one merge.


## 11. 7-row rolling mean and rank within group

In [12]:
df_sorted = df.sort_values(['city','sale_date']).copy()
df_sorted['median_roll7'] = (df_sorted.groupby('city', observed=True)['median']
                             .transform(lambda s: s.rolling(7, min_periods=1).mean()))
df_sorted['sales_rank_in_city'] = (df_sorted.groupby('city', observed=True)['sales']
                                   .rank(method='dense', ascending=False))
print(df_sorted[['city','sale_date','median','median_roll7','sales','sales_rank_in_city']].head(15))

       city  sale_date   median  median_roll7  sales  sales_rank_in_city
0   Abilene 2000-01-01  71400.0  71400.000000   72.0               109.0
1   Abilene 2000-02-01  58700.0  65050.000000   98.0                99.0
2   Abilene 2000-03-01  58100.0  62733.333333  130.0                74.0
3   Abilene 2000-04-01  68600.0  64200.000000   98.0                99.0
4   Abilene 2000-05-01  67300.0  64820.000000  141.0                63.0
5   Abilene 2000-06-01  66900.0  65166.666667  156.0                48.0
6   Abilene 2000-07-01  73500.0  66357.142857  152.0                52.0
7   Abilene 2000-08-01  75000.0  66871.428571  131.0                73.0
8   Abilene 2000-09-01  64500.0  67700.000000  104.0                95.0
9   Abilene 2000-10-01  59300.0  67871.428571  101.0                96.0
10  Abilene 2000-11-01  70900.0  68200.000000  100.0                97.0
11  Abilene 2000-12-01  65000.0  67871.428571   92.0               104.0
12  Abilene 2001-01-01  64500.0  67528.571429   75.

## 12. Three suspicious rows

In [13]:
suspects = pd.concat([
    df.loc[df['listings'].idxmin()].to_frame().T,
    df.loc[df['inventory'].idxmax()].to_frame().T,
    df.loc[df['volume'].idxmax()].to_frame().T,
])
suspects[['city','sale_date','sales','volume','median','listings','inventory']]

,city,sale_date,sales,volume,median,listings,inventory
5357,Midland,2010-02-01 00:00:00,100.0,19880017.0,175400.0,0.0,0.0
7423,South Padre Island,2010-11-01 00:00:00,10.0,2098500.0,160000.0,1280.0,55.900002
3739,Houston,2015-07-01 00:00:00,8945.0,2568156780.0,217600.0,23875.0,3.4


- **Minimum listings row:** a reported active-listing count of 0 alongside a nonzero market is suspicious and may be a source/reporting gap.
- **Maximum inventory row:** 55.9 months of inventory is an extreme tail value and should be verified against the source.
- **Maximum volume row:** the largest monthly dollar volume is valid-looking but is a leverage point that can dominate means and plots.

## 13. CSV vs Parquet size and load time

In [14]:
cleaned = df.copy()
cleaned_csv = Path('cleaned_txhousing.csv')
cleaned_parquet = Path('cleaned_txhousing.parquet')
cleaned.to_csv(cleaned_csv, index=False)

csv_t0 = time.perf_counter(); _ = pd.read_csv(cleaned_csv); csv_load_s = time.perf_counter() - csv_t0
csv_size = cleaned_csv.stat().st_size

try:
    import pyarrow  # noqa: F401
    parquet_ready = True
except ImportError:
    # Colab normally has pyarrow. If this runtime does not, install it automatically there.
    import sys, subprocess
    if 'google.colab' in sys.modules:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
        import pyarrow  # noqa: F401
        parquet_ready = True
    else:
        parquet_ready = False

if parquet_ready:
    cleaned.to_parquet(cleaned_parquet, index=False)
    pq_t0 = time.perf_counter(); _ = pd.read_parquet(cleaned_parquet); pq_load_s = time.perf_counter() - pq_t0
    pq_size = cleaned_parquet.stat().st_size
    pq_saving = (1 - pq_size / csv_size) * 100
    print(f'CSV:     {csv_size:,} bytes, load {csv_load_s:.6f} s')
    print(f'Parquet: {pq_size:,} bytes, load {pq_load_s:.6f} s')
    print(f'Parquet size saving: {pq_saving:.2f}%')

    # Keep README numbers synchronized after Restart -> Run all in Colab.
    readme_path = Path('README.md')
    if readme_path.exists():
        import re
        text = readme_path.read_text(encoding='utf-8')
        text = re.sub(r'- Parquet size: .*', f'- Parquet size: **{pq_size:,} bytes**', text)
        readme_path.write_text(text, encoding='utf-8')
else:
    print(f'CSV: {csv_size:,} bytes, load {csv_load_s:.6f} s')
    print('No Parquet engine is installed in this local runtime.')
    print('When run in Google Colab, this cell installs pyarrow if needed, writes the Parquet file,')
    print('prints the exact Parquet size/load time, and updates README.md automatically.')


CSV:     900,524 bytes, load 0.011996 s
Parquet: 183,789 bytes, load 0.324417 s
Parquet size saving: 79.59%


## Submission check

In [15]:
print('Rows:', len(df), 'Columns:', df.shape[1])
print('Required source CSV present:', DATA_PATH.exists())
print(f'Memory saving to report in README: {mem_saving_pct:.2f}%')
print('Run Runtime -> Restart and run all in Colab before submission.')

Rows: 8602 Columns: 16
Required source CSV present: True
Memory saving to report in README: 58.92%
Run Runtime -> Restart and run all in Colab before submission.
